# Name That LLM — submission notebook

18-class attribution with sparse stylometry. **Offline by design**: no network
calls, no downloads, no API keys.

Inputs required on the right-hand panel:
1. the competition dataset (`name-that-llm`), and
2. the prediction-artifacts dataset built with
   `python -m src.cli package --kaggle-user <your-username>` (contains
   `manifest.json`, `models/`, optional `weights.json`, and the `src/` package
   needed to unpickle the bundles).

Output: `submission.parquet` in the working directory, schema `id, model`.

In [ ]:
from __future__ import annotations

import glob
import json
import sys
from pathlib import Path

import pandas as pd

ARTIFACT_CANDIDATES = [
    p for p in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
]
if not ARTIFACT_CANDIDATES:
    raise FileNotFoundError(
        "No manifest.json found under /kaggle/input. Attach the artifacts dataset "
        "created by `python -m src.cli package`."
    )
ARTIFACT_DIR = Path(ARTIFACT_CANDIDATES[0]).parent
sys.path.insert(0, str(ARTIFACT_DIR))          # lets `src` be importable
sys.path.insert(0, str(ARTIFACT_DIR / "src"))  # fallback layout

from src.constants import LABELS, N_CLASSES  # noqa: E402
from src.inference import load_final, predict_dataframe  # noqa: E402
from src.submission import build_submission  # noqa: E402

print(f"artifacts: {ARTIFACT_DIR}")
print(f"classes: {N_CLASSES}")

In [ ]:
TEST_PATHS = [
    Path("/kaggle/input/competitions/name-that-llm/test.csv"),
    Path("/kaggle/input/name-that-llm/test.csv"),
    Path("data/synthetic/test.csv"),  # local smoke test only
]
test_path = next((p for p in TEST_PATHS if p.is_file()), None)
if test_path is None:
    raise FileNotFoundError("could not locate test.csv in any known input path")

test = pd.read_csv(test_path)
assert {"id", "text"} <= set(test.columns), f"unexpected columns: {list(test.columns)}"
assert test["id"].is_unique, "test.csv has duplicate ids"
assert test["text"].notna().all(), "test.csv has null text"
print(f"test rows: {len(test)} from {test_path}")

In [ ]:
bundles, manifest = load_final(ARTIFACT_DIR)
assert manifest["class_order"] == list(LABELS), "artifact class order mismatch"

weights_path = ARTIFACT_DIR / "weights.json"
weights = None
if weights_path.is_file():
    payload = json.loads(weights_path.read_text(encoding="utf-8"))
    weights = {k: float(v) for k, v in payload.get("weights", payload).items() if float(v) > 0}
    weights = {k: v for k, v in weights.items() if k in bundles}
    if not weights:
        weights = None

print("models:", sorted(bundles))
print("blend weights:", weights or "equal weights")

predictions = predict_dataframe(bundles, test, weights)
assert len(predictions) == len(test)
assert set(predictions) <= set(LABELS)
print(pd.Series(predictions).value_counts().head())

In [ ]:
out = build_submission(
    test["id"], predictions, "submission.parquet", expected_ids=test["id"]
)
check = pd.read_parquet(out)
assert list(check.columns) == ["id", "model"]
assert len(check) == len(test)
assert check["id"].is_unique and not check.isna().any().any()
assert check["model"].isin(LABELS).all()
print(f"wrote {out} with {len(check)} rows")
check.head()

## Notes

- Internet is disabled for this competition; the notebook never fetches
  weights, models or data at runtime.
- `submission.parquet` is written to the working directory, as required.
- The class order is fixed by the artifact manifest and checked against
  `src.constants.LABELS`; a mismatched artifact fails loudly instead of
  silently permuting labels.
- If this notebook is run on the synthetic fixture (`data/synthetic/test.csv`),
  the output is a pipeline check, not a competition submission.